# Credit Card Default Prediction — Training Notebook

**Stage 1.5** of the project (see `docs/progress.md`). Run this in Google Colab.

Dataset: [Default of Credit Card Clients](https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset)

Workflow reminder (full detail in `docs/mlflow-workflow.md`):
1. Everything here logs to a **local** MLflow tracking URI (`file:./mlruns`) — no network setup needed from Colab.
2. At the end of the session, zip `mlruns/` and download it, along with the best model's artifacts.
3. Drop `mlruns/` into `infra/mlflow/data/mlruns` on your machine to browse everything in the local MLflow UI.
4. Drop the exported model + preprocessing artifacts into `ml/artifacts/` for the FastAPI backend (Stage 2).

## 0. Setup

In [ ]:
!pip install -q "mlflow==2.16.2" torch scikit-learn imbalanced-learn pandas matplotlib seaborn optuna

# Pinned to 2.16.2 on purpose — matches infra/mlflow/Dockerfile exactly.
#
# Why this matters: `pip install mlflow` unpinned pulls MLflow 3.x, which put the
# FileStore backend (the plain-folder `mlruns/`, no database) into "maintenance mode" —
# `mlflow.set_experiment(...)` raises MlflowException unless you opt back in with
# MLFLOW_ALLOW_FILE_STORE=true. Setting that env var would silence the error, but it
# doesn't guarantee the on-disk format 3.x writes is byte-for-byte what 2.16.2 (running
# in the Docker server) reads back. Pinning both sides to the same version removes that
# risk entirely — this is the fix if you hit:
#   MlflowException: The filesystem tracking backend (e.g., './mlruns') is in
#   maintenance mode ...

import mlflow

# Local file-store tracking — matches the FileStore backend used by the
# Dockerized MLflow server, so mlruns/ can be copied over directly later.
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("credit-card-default")

## 1. Load Data

Download the dataset from Kaggle (via `kagglehub` or manual upload) into `ml/data/` conventions — in Colab, just load it into a DataFrame directly.

In [ ]:
# --- Option A (default): manual upload ---------------------------------------
# Download the CSV once from Kaggle (button on the dataset page), then run this
# cell and pick the file when the widget appears. No Kaggle account/API token
# juggling inside Colab — one less thing to debug on day one.
from google.colab import files

uploaded = files.upload()  # pick the downloaded CSV in the dialog
csv_filename = next(iter(uploaded))  # grabs whatever filename you uploaded

# --- Option B: kagglehub (uncomment if you'd rather not re-upload each session) ---
# import kagglehub
# path = kagglehub.dataset_download("uciml/default-of-credit-card-clients-dataset")
# csv_filename = f"{path}/UCI_Credit_Card.csv"

import pandas as pd

df = pd.read_csv(csv_filename)

# Quirk #1: the ID column is a row identifier, not a feature — drop it.
df = df.drop(columns=["ID"])

# Quirk #2 (easy to miss): the repayment-status columns are named
# PAY_0, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6 — there is no PAY_1. That's not a
# typo in this notebook, it's how the original dataset is labeled. PAY_0 is
# the most recent month, PAY_6 the oldest.

print(df.shape)
df.head()

## 2. Exploratory Data Analysis

- Target distribution (`default.payment.next.month`) — quantify the class imbalance
- Univariate distributions: `LIMIT_BAL`, `AGE`, `BILL_AMT1-6`, `PAY_AMT1-6`
- `PAY_0..PAY_6` repayment status patterns vs. default
- Data quality: undocumented codes in `EDUCATION` (0, 5, 6) and `MARRIAGE` (0)
- Correlation heatmap

### 2.1 Target distribution — how imbalanced is this, exactly?

The first number to establish, because it determines which metrics are trustworthy for the rest of the
notebook.

In [ ]:
# 2.1 Target distribution — how imbalanced are we actually dealing with?
target = "default.payment.next.month"

counts = df[target].value_counts().sort_index()
pct = df[target].value_counts(normalize=True).sort_index() * 100

print("Class counts:\n", counts)
print("\nClass %:\n", pct.round(2))

# The number that matters most for everything downstream: if you did nothing
# clever and just predicted "no default" for every single customer, this is
# the accuracy you'd get for free — with zero predictive power.
naive_accuracy = counts[0] / counts.sum()
print(f"\nNaive 'always predict no-default' accuracy: {naive_accuracy:.2%}")
print("Any model has to beat THIS meaningfully on precision/recall — not just accuracy.")

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=["No Default (0)", "Default (1)"], y=counts.values, ax=ax, palette=["#4C72B0", "#C44E52"])
for i, v in enumerate(counts.values):
    ax.text(i, v + 200, f"{v}\n({pct.values[i]:.1f}%)", ha="center")
ax.set_ylabel("Number of customers")
ax.set_title("Target distribution — default.payment.next.month")
plt.show()

**Implication:** ~78/22 split — not extreme (like fraud's 99.9/0.1), but enough that accuracy is a
misleading headline metric. A model that just memorizes the majority class scores ~78% "accuracy" while
being useless. This is *why* Section 4 (Handling Class Imbalance) and the metrics we track from here on
(precision, recall, F1, PR-AUC) exist — keep this naive-accuracy number around as the floor everything else
has to clear.

### 2.2 Missing values & duplicates

In [ ]:
# 2.2 Missing values & duplicates — quick hygiene check before trusting anything else
print("Missing values per column (top 5, should be all 0 for this dataset):")
print(df.isnull().sum().sort_values(ascending=False).head())

print(f"\nExact duplicate rows: {df.duplicated().sum()}")

# Not glamorous, but skipping this step and finding out three sections later
# that you had NaNs silently propagating through a scaler is a worse afternoon.

### 2.3 Data quality: undocumented category codes

The dataset's documentation defines:
- `EDUCATION`: 1=graduate school, 2=university, 3=high school, 4=others
- `MARRIAGE`: 1=married, 2=single, 3=others

But real values in the columns don't stop there — let's look.

In [ ]:
print("EDUCATION value counts:")
print(df["EDUCATION"].value_counts().sort_index())
# -> you'll see 0, 5, 6 show up too — undocumented codes, almost certainly
#    data-entry artifacts or an "unknown" bucket that never made it into the docs.

print("\nMARRIAGE value counts:")
print(df["MARRIAGE"].value_counts().sort_index())
# -> 0 shows up here too, outside the documented 1/2/3.

# Not fixing these here — this is EDA, we're only *finding* the issue. The fix
# (folding 0/5/6 into EDUCATION's "others"=4, and 0 into MARRIAGE's "others"=3)
# happens in Section 3, where every preprocessing decision belongs together.

### 2.4 Numeric distributions — shape matters for a neural net

MLPs are sensitive to feature scale (that's *why* Section 3 ends in `StandardScaler`, in 3.6), and gradient
descent behaves badly when a feature is heavily right-skewed with a long tail of outliers — a few whales
with huge bill amounts can dominate the loss early in training. Let's see which columns actually look like
that before deciding anything.

In [ ]:
# LIMIT_BAL (credit limit) and AGE — the two "plain" continuous features
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df["LIMIT_BAL"], bins=40, ax=axes[0], color="#4C72B0")
axes[0].set_title("LIMIT_BAL distribution")
sns.histplot(df["AGE"], bins=40, ax=axes[1], color="#55A868")
axes[1].set_title("AGE distribution")
plt.tight_layout()
plt.show()

print("LIMIT_BAL skew:", df["LIMIT_BAL"].skew().round(2))
print("AGE skew:", df["AGE"].skew().round(2))

# BILL_AMT1 / PAY_AMT1 — representative of the 6 bill/payment columns each.
# Plotting raw vs log1p side-by-side to make the skew concrete rather than
# just quoting a skew number.
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
sns.histplot(df["BILL_AMT1"], bins=50, ax=axes[0, 0], color="#C44E52")
axes[0, 0].set_title(f"BILL_AMT1 (raw) — skew={df['BILL_AMT1'].skew():.2f}")
sns.histplot(np.sign(df["BILL_AMT1"]) * np.log1p(np.abs(df["BILL_AMT1"])), bins=50, ax=axes[0, 1], color="#C44E52")
axes[0, 1].set_title("BILL_AMT1 (signed log1p) — for comparison only")

sns.histplot(df["PAY_AMT1"], bins=50, ax=axes[1, 0], color="#8172B2")
axes[1, 0].set_title(f"PAY_AMT1 (raw) — skew={df['PAY_AMT1'].skew():.2f}")
sns.histplot(np.log1p(df["PAY_AMT1"]), bins=50, ax=axes[1, 1], color="#8172B2")
axes[1, 1].set_title("PAY_AMT1 (log1p) — for comparison only")
plt.tight_layout()
plt.show()

# Note: BILL_AMT can be negative (overpayment/credit balance), which is why the
# signed-log trick is used instead of a plain log1p — plain log1p breaks on
# negative input. This plot is just showing you the shape difference; whether
# we actually apply a log transform is a Section 3 decision — and it is
# applied, in 3.4, after the engineered ratio features are built from the
# raw values (a ratio of two logs is not the log of a ratio).

**Reading these plots:** `LIMIT_BAL` and `AGE` are both right-skewed but mildly — no transform needed,
`StandardScaler` in Section 3 handles this fine. `BILL_AMT1` and `PAY_AMT1` are a different story: the raw
histograms (left column) are a tall spike near zero with a long thin tail stretching far right — most
customers carry small balances/payments, a few carry very large ones. The log1p versions (right column)
pull that tail in and spread the bulk of the mass out, which is closer to the "well-behaved" shape gradient
descent likes. We're not applying the log transform yet — just confirming *why* it's on the table for
Section 3, alongside the ratio features that address the same skew a different way.

### 2.5 Repayment status (`PAY_0..PAY_6`) vs. default — the headline signal

`PAY_0` is last month's repayment status: -1/0 roughly mean "paid on time / revolving credit used
properly", and 1, 2, 3... mean "N months late". If there's one relationship in this dataset that should be
strong, it's this one — let's check, and use it as the sanity check for everything else: if a supposedly
"engineered" feature in Section 3 correlates with default *less* than raw `PAY_0`, that's a signal the
engineering didn't add much.

In [ ]:
pay_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, pay_cols):
    default_rate_by_status = df.groupby(col)[target].mean().sort_index()
    n_by_status = df.groupby(col).size().sort_index()
    sns.barplot(x=default_rate_by_status.index, y=default_rate_by_status.values, ax=ax, color="#C44E52")
    ax.axhline(naive_accuracy_complement := df[target].mean(), color="gray", linestyle="--", linewidth=1)
    ax.set_title(col)
    ax.set_ylabel("Default rate")
    ax.set_xlabel("Repayment status code")
plt.suptitle("Default rate by repayment status, per month (dashed line = overall default rate)", y=1.02)
plt.tight_layout()
plt.show()

# Reading this: for PAY_0, status <= 0 (paid on time) sits near/below the dashed
# overall-rate line; each step up in "months late" pushes the default rate up,
# often steeply. That monotonic climb is exactly the kind of pattern a linear
# model AND a neural net can both exploit easily — this is your strongest
# individual predictor before any feature engineering happens at all.

**Reading this plot:** for `PAY_0` especially, bars at status ≤ 0 (paid on time / no consumption) sit at or
below the dashed overall-default-rate line, then climb — often sharply — as the status code increases (more
months late). The same shape repeats, a bit weaker, across `PAY_2` through `PAY_6`, and it fades slightly
the further back in time you go (recent behavior predicts next-month default better than 6-month-old
behavior — makes intuitive sense). Concretely: **`PAY_0` is the single most useful raw column in this
dataset**, which is exactly what the ranked correlation list in 2.6 will confirm numerically.

### 2.6 Correlation heatmap — which raw features already "know" the target?

A heatmap won't catch non-linear relationships (a neural net can), but it's a fast way to see which raw
columns already carry signal and which look like noise before you've engineered anything. Worth revisiting
after Section 3 — if an engineered feature doesn't beat its raw ingredients here, question whether it earns
its place in the model.

In [ ]:
corr = df.corr(numeric_only=True)

# Full heatmap is dense (23 features) — also print just the target correlations,
# sorted, since that ranked list is usually more actionable than eyeballing a grid.
target_corr = corr[target].drop(target).sort_values(key=abs, ascending=False)
print("Features ranked by |correlation| with default:")
print(target_corr.round(3))

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, ax=ax)
ax.set_title("Correlation matrix — all numeric features")
plt.tight_layout()
plt.show()

# Expect PAY_0..PAY_6 to dominate the top of that ranked list (confirms 2.5),
# LIMIT_BAL to show a modest negative correlation (higher credit limit ~ lower
# default, likely because it's a proxy for creditworthiness the bank already
# assessed), and the six BILL_AMT columns to correlate strongly WITH EACH
# OTHER (multicollinearity — a customer's bill in month 1 is similar to month
# 2) more than with the target individually. That last point is itself useful:
# six highly-correlated raw columns is a candidate for feature engineering
# (e.g. a trend/slope feature) instead of feeding all six in raw.

**Reading this plot:** the ranked list will show `PAY_0..PAY_6` at the top (confirming 2.5 numerically —
`PAY_0` typically lands around |r| ≈ 0.3-0.4, the strongest of any raw column), `LIMIT_BAL` with a modest
*negative* correlation (banks already priced in creditworthiness when they set the limit, so it's a weak
proxy for the same thing the model is trying to predict), and most `BILL_AMT*`/demographic columns clustered
near zero — individually weak. On the heatmap itself, look for the bright block among `BILL_AMT1..6`: they
correlate strongly with **each other** (0.8+), not just the target. That's multicollinearity, and it's the
concrete evidence behind the "6 columns → 1 trend feature" idea from Section 2.4/2.8.

### 2.7 Default rate across demographic slices

Not because these will necessarily be strong predictors (they usually aren't, compared to `PAY_0`), but
because it's worth knowing whether the model's errors will skew across sex/education/marital-status groups
before it's deployed — that's an error-analysis and fairness question worth having eyes on early, not
something to discover after Section 9.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sex_labels = {1: "Male", 2: "Female"}
sns.barplot(x=df["SEX"].map(sex_labels), y=df[target], ax=axes[0], color="#4C72B0", errorbar=None)
axes[0].set_title("Default rate by SEX")
axes[0].set_ylabel("Default rate")

sns.barplot(x=df["EDUCATION"], y=df[target], ax=axes[1], color="#55A868", errorbar=None)
axes[1].set_title("Default rate by EDUCATION (raw codes incl. undocumented)")

sns.barplot(x=df["MARRIAGE"], y=df[target], ax=axes[2], color="#8172B2", errorbar=None)
axes[2].set_title("Default rate by MARRIAGE (raw codes incl. undocumented)")

plt.tight_layout()
plt.show()

# AGE as a binned view — raw age is noisy, bucketing makes any trend visible
age_bins = pd.cut(df["AGE"], bins=[20, 30, 40, 50, 60, 80], labels=["21-30", "31-40", "41-50", "51-60", "60+"])
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=age_bins, y=df[target], ax=ax, color="#C44E52", errorbar=None)
ax.set_title("Default rate by age bracket")
ax.set_ylabel("Default rate")
plt.show()

**Reading these plots:** expect small, noisy differences rather than a dramatic signal — e.g. `EDUCATION`'s
undocumented codes (0, 5, 6) often show wildly different bars simply because they have very few rows behind
them (low n, high variance, not a real effect — another reason to fold them into "others" in Section 3
rather than trust them as their own category). The age-bracket view usually shows default rate ticking up
slightly at both tails (younger, less credit history / older, fixed income) with a dip in the middle
brackets. None of this rivals `PAY_0` in strength, but it's worth remembering when Section 9's error
analysis looks at whether misclassifications cluster in any of these groups.

### 2.8 EDA summary → what it decides for Section 3

| Finding | Decision it drives |
|---|---|
| ~78/22 class split | Track precision/recall/F1/PR-AUC everywhere, not accuracy; revisit imbalance handling in Section 4 |
| `EDUCATION`/`MARRIAGE` have undocumented codes (0, 5, 6 / 0) | Consolidate into the documented "others" bucket before encoding |
| `BILL_AMT*`/`PAY_AMT*` are heavily right-skewed | `StandardScaler` alone may not be enough — consider engineered ratio/trend features that are naturally less skewed than the raw amounts |
| `PAY_0..PAY_6` dominate correlation with target, and climb monotonically with lateness | Strongest raw signal — any engineered feature (delinquency streak, etc.) should be judged against beating this baseline, not replacing it |
| `BILL_AMT1..6` are highly collinear with each other | A trend/slope feature across the 6 months may capture more than 6 raw correlated columns |
| Demographic slices (`SEX`/`EDUCATION`/`MARRIAGE`/`AGE`) show smaller, noisier effects than `PAY_0` | Still worth encoding as features, but keep an eye on them in Section 9's error analysis for skewed error rates |

Nothing gets fixed in this section on purpose — EDA's job is to *decide* what Section 3 needs to do and
*why*, not to do it. Next: Section 3, acting on this table.

## 3. Preprocessing & Feature Engineering

Acting on the decision table from 2.8. Everything here is a *decision with a reason attached* — nothing is
transformed just because it's conventional.

### 3.1 Consolidate the undocumented category codes

In [ ]:
# 3.1 Fix the undocumented category codes found in EDA 2.3
df_fe = df.copy()

# EDUCATION: fold 0, 5, 6 (undocumented) into 4 ("others") — the documented bucket
# that already means "doesn't fit graduate/university/high-school".
df_fe["EDUCATION"] = df_fe["EDUCATION"].replace({0: 4, 5: 4, 6: 4})

# MARRIAGE: fold 0 (undocumented) into 3 ("others").
df_fe["MARRIAGE"] = df_fe["MARRIAGE"].replace({0: 3})

print("EDUCATION after cleanup:", sorted(df_fe["EDUCATION"].unique()))
print("MARRIAGE after cleanup:", sorted(df_fe["MARRIAGE"].unique()))

### 3.2 Feature engineering

Six new features, each chosen to address a specific thing EDA flagged rather than engineered for its own
sake:

| Feature | What it captures | Why (from EDA) |
|---|---|---|
| `PAY_AVG` | Mean repayment status across the 6 months | Smooths single-month noise out of the strongest raw signal (2.5) |
| `PAY_MAX` | Worst (most-late) status in the 6 months | "Ever seriously late" can matter more than "late on average" |
| `DELINQUENCY_STREAK` | Count of months with status > 0 (late) | A repeat-late customer reads differently than one bad month |
| `BILL_TREND` | Slope of a linear fit across `BILL_AMT1..6` | Replaces 6 collinear columns (2.6) with one "balance rising/falling" number |
| `AVG_BILL` | Mean bill amount across 6 months | Also collapses the 6 collinear `BILL_AMT*` columns |
| `UTILIZATION` | `AVG_BILL / LIMIT_BAL` | Classic credit-risk feature: % of credit limit typically used |
| `PAY_TO_BILL_RATIO` | `AVG_PAY_AMT / AVG_BILL` | Are they paying down what they owe, or barely touching it? |

In [ ]:
pay_status_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
pay_amt_cols = [f"PAY_AMT{i}" for i in range(1, 7)]

# --- Repayment-status aggregates ---
df_fe["PAY_AVG"] = df_fe[pay_status_cols].mean(axis=1)
df_fe["PAY_MAX"] = df_fe[pay_status_cols].max(axis=1)
df_fe["DELINQUENCY_STREAK"] = (df_fe[pay_status_cols] > 0).sum(axis=1)

# --- Bill trend: slope of a straight-line fit across the 6 months ---
# BILL_AMT1 is the most recent month, BILL_AMT6 the oldest — reverse so the
# fit reads chronologically (oldest -> newest), so a positive slope means
# "balance climbing toward the target month", which is the intuitive
# direction to reason about.
bill_matrix_chronological = df_fe[bill_cols[::-1]].to_numpy()
months = np.arange(6)


def linear_slope(row: np.ndarray) -> float:
    return np.polyfit(months, row, 1)[0]


df_fe["BILL_TREND"] = np.apply_along_axis(linear_slope, 1, bill_matrix_chronological)

# --- Utilization & payment-to-bill ratio ---
# NOTE: these are computed from the RAW monetary columns, before the optional
# log transform below — a ratio of two log-transformed numbers is not the same
# quantity as the log of a ratio, and "% of credit limit used" is only
# meaningful on the raw scale.
df_fe["AVG_BILL"] = df_fe[bill_cols].mean(axis=1)
avg_pay_amt = df_fe[pay_amt_cols].mean(axis=1)

# LIMIT_BAL is never 0 in this dataset, but guard anyway rather than trust that blindly.
df_fe["UTILIZATION"] = (df_fe["AVG_BILL"] / df_fe["LIMIT_BAL"].replace(0, np.nan)).fillna(0)
# Utilization can go negative (overpayment) or spike very high for edge-case rows —
# clip to a sane range so a handful of outliers don't dominate the scaled feature.
df_fe["UTILIZATION"] = df_fe["UTILIZATION"].clip(-1, 3)

# AVG_BILL can be 0 or negative (no/overpaid balance) — guard the ratio the same way.
df_fe["PAY_TO_BILL_RATIO"] = (avg_pay_amt / df_fe["AVG_BILL"].replace(0, np.nan)).fillna(0)
df_fe["PAY_TO_BILL_RATIO"] = df_fe["PAY_TO_BILL_RATIO"].clip(-5, 5)

engineered_cols = ["PAY_AVG", "PAY_MAX", "DELINQUENCY_STREAK", "BILL_TREND", "AVG_BILL", "UTILIZATION", "PAY_TO_BILL_RATIO"]
df_fe[engineered_cols].describe()

### 3.3 Did the engineering actually help? Checking before we trust it

Same test EDA 2.6 promised: rank the new features by |correlation| with the target, and see where they
land next to the raw columns that fed them. If `PAY_AVG`/`PAY_MAX` don't come close to raw `PAY_0`, or
`BILL_TREND`/`UTILIZATION` don't beat the individual raw `BILL_AMT*` columns, that's a real finding too —
not everything engineered earns its keep, and it's better to find that out now than after training.

In [ ]:
compare_cols = ["PAY_0", "PAY_2", "BILL_AMT1", "LIMIT_BAL"] + engineered_cols
corr_compare = df_fe[compare_cols + [target]].corr(numeric_only=True)[target].drop(target)
corr_compare = corr_compare.reindex(corr_compare.abs().sort_values(ascending=False).index)

# Color raw columns differently from engineered ones so the comparison is visual, not just a table.
colors = ["#4C72B0" if c in engineered_cols else "#999999" for c in corr_compare.index]

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=corr_compare.values, y=corr_compare.index, palette=colors, ax=ax)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlation with default")
ax.set_title("Raw (gray) vs. engineered (blue) features — correlation with target")
plt.tight_layout()
plt.show()

print(corr_compare.round(3))

**What the run actually showed** (your Colab output, correlation with the target):

| Feature | \|r\| | Verdict |
|---|---|---|
| `DELINQUENCY_STREAK` | **0.398** | 🏆 Beats every raw column, including `PAY_0` (0.325), by ~22% |
| `PAY_MAX` | 0.331 | Slightly beats `PAY_0` |
| `PAY_0` (raw) | 0.325 | The raw benchmark to beat |
| `PAY_AVG` | 0.282 | Below `PAY_0` — averaging smoothed away signal rather than adding any |
| `UTILIZATION` | 0.116 | Beats **every** `BILL_AMT*` column (all ≤ 0.020) by roughly 6× |
| `PAY_TO_BILL_RATIO` | -0.081 | Weak but 4× better than raw `BILL_AMT1` |
| `BILL_TREND` | -0.024 | ❌ Failed — no better than the raw columns it was built from |

**The headline result:** `DELINQUENCY_STREAK` — literally just "how many of the 6 months was this customer
late?" — is the strongest single predictor in the dataset, stronger than any raw column the data shipped
with. That is feature engineering earning its place, measurably, before a single epoch of training. The
reason it beats `PAY_0` is that `PAY_0` only sees last month; a customer late 4 months running and a
customer late once look identical to `PAY_0` but very different to `DELINQUENCY_STREAK`.

**The honest failure:** `BILL_TREND` (-0.024) did not work. The theory in 3.2 — that the *slope* of the
balance over 6 months carries signal the collinear raw columns don't — simply isn't supported here. A
rising balance is apparently about as common among people who repay as among those who default. It stays
in the feature set (it costs almost nothing and an MLP may still combine it with other features
non-linearly), but do **not** claim it as a win in the writeup. Reporting an engineered feature that
didn't pan out is part of doing this properly.

**Note on `PAY_AVG` vs `PAY_MAX`:** both are built from the same six columns, but `PAY_MAX` (0.331) beats
`PAY_AVG` (0.282). "Worst month" carries more risk information than "typical month" — one serious
delinquency matters more than a mildly elevated average. Small detail, but it's exactly the kind of thing
you only learn by measuring both instead of picking one.

### 3.4 The log transform EDA 2.4 flagged — applied now, deliberately

Section 2.4 measured the skew on `BILL_AMT*` / `PAY_AMT*`, plotted raw vs. log side by side, and said the
decision belonged to Section 3 — and then the first version of this notebook quietly never made it. Making
it explicitly now.

**Why it should help:** `StandardScaler` (coming in 3.6) only shifts and rescales — it does **not** change
distribution *shape*. A column with skew ≈ 3 is still skew ≈ 3 after scaling, so a handful of very large
bills still produce very large activations in the first layer and dominate early gradients. `signed log1p`
compresses that tail while preserving sign, since `BILL_AMT` can legitimately be negative (an overpaid
account).

**Why it happens *here*, after 3.3 and not before:** the ratio features in 3.2 (`UTILIZATION`,
`PAY_TO_BILL_RATIO`) are computed from the raw monetary values — a ratio of two logs is not the log of a
ratio, and "% of credit limit used" is only meaningful on the raw scale. And 3.3's raw-vs-engineered
comparison has to be against genuinely raw columns to mean anything. So: engineer, compare, *then*
transform.

`APPLY_LOG_TRANSFORM` is a flag — set it to `False`, re-run from here, and compare the final metrics to see
for yourself whether it earned its place on this dataset.

In [ ]:
# NEW in this revision — see the markdown above for why it sits here and not in 3.2.
APPLY_LOG_TRANSFORM = True

if APPLY_LOG_TRANSFORM:
    def signed_log1p(s):
        """Plain np.log1p breaks on negative input; BILL_AMT can be negative
        (overpayment / credit balance), so compress magnitude but keep the sign."""
        return np.sign(s) * np.log1p(np.abs(s))

    skew_before = df_fe[bill_cols + pay_amt_cols].skew()
    for c in bill_cols + pay_amt_cols:
        df_fe[c] = signed_log1p(df_fe[c])
    skew_after = df_fe[bill_cols + pay_amt_cols].skew()

    comparison = pd.DataFrame({"skew_before": skew_before.round(2), "skew_after": skew_after.round(2)})
    comparison["improved"] = comparison["skew_after"].abs() < comparison["skew_before"].abs()
    print(comparison)
    print(f"\nColumns whose skew moved closer to 0: {comparison['improved'].sum()} of {len(comparison)}")
else:
    print("Log transform skipped (APPLY_LOG_TRANSFORM = False) — monetary columns left on their raw scale.")

### 3.5 Encode categoricals, then split (before scaling — see the note in the next cell)

In [ ]:
# SEX is already binary (1=male, 2=female) — remap to 0/1 rather than one-hot,
# no information gained from a second dummy column for a 2-level variable.
df_fe["SEX"] = df_fe["SEX"].map({1: 0, 2: 1})

# EDUCATION and MARRIAGE are nominal (no inherent order), even though they're
# stored as ints — one-hot them so the model doesn't invent a false ordering
# (e.g. "high school < university" implying a linear relationship that isn't there).
df_encoded = pd.get_dummies(df_fe, columns=["EDUCATION", "MARRIAGE"], drop_first=True, dtype=int)

feature_columns = [c for c in df_encoded.columns if c != target]
print(f"Final feature count: {len(feature_columns)}")
print(feature_columns)

### 3.6 Stratified train / validation / test split, then scale

**Split before scaling, always.** `StandardScaler` learns a mean and standard deviation from whatever data
you fit it on. Fit it on the full dataset and those statistics have quietly seen the validation/test rows —
a subtle form of data leakage that makes validation metrics look slightly better than what the model will
actually see in production. Fitting only on `X_train` and *applying* (`.transform`, never `.fit` again) to
val/test keeps the held-out sets honest.

Split is 70/15/15 (train/val/test), stratified on the target both times so the ~22% default rate from EDA
2.1 is preserved in all three sets — otherwise a random split could hand you an unrepresentative validation
set purely by chance, given how much data is in the minority class.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

X = df_encoded[feature_columns]
y = df_encoded[target]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print("\nDefault rate per split (should all be ~22%, confirming stratify worked):")
for name, y_split in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"  {name}: {y_split.mean():.2%}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit ONLY on train
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# feature_columns (order!) + scaler are both needed, unchanged, at inference time
# in the FastAPI backend — this exact list/order is what gets exported in Section 10.
print(f"\nfeature_columns is now the contract the backend must match: {len(feature_columns)} columns, in this order.")

## 4. Handling Class Imbalance

Four strategies, compared fairly, then one carried forward into every later section.

### 4.1 Shared infrastructure: model, training loop, evaluation

Comparing imbalance strategies means training the same model four different ways — so the reusable
architecture and training loop have to exist *now*, even though Section 5 is where the architecture's design
choices get documented. The class defined here is the same one used, unchanged, through Section 9.

**Three things changed in this cell** compared to the first version of this notebook:

1. **Per-epoch MLflow logging.** `train_model` now calls `mlflow.log_metrics(..., step=epoch)` every epoch,
   so the MLflow UI draws real training curves — train loss, val loss, and validation precision / recall /
   F1 / PR-AUC over time — instead of storing one final number per run. This is what makes "watch precision
   and recall evolve during training" possible in the MLflow UI.
2. **`history` carries validation metrics per epoch**, not just losses, so any cell below can plot them.
3. **`monitor` lets early stopping watch something other than `val_loss`.** This matters more than it
   sounds — see 9.1.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
)

torch.manual_seed(RANDOM_STATE)


class CreditDefaultMLP(nn.Module):
    """Configurable MLP. Every constructor argument here becomes a field in
    model_config.json (Section 10) — the FastAPI backend rebuilds this exact
    class from that file before loading the trained weights, so nothing here
    is allowed to be "just a default I'll remember"."""

    def __init__(self, input_dim, hidden_dims=(64, 32), dropout=0.2, use_batchnorm=True, init_scheme="he"):
        super().__init__()
        self.config = {
            "input_dim": input_dim,
            "hidden_dims": list(hidden_dims),
            "dropout": dropout,
            "use_batchnorm": use_batchnorm,
            "init_scheme": init_scheme,
        }
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h
        # Single output logit — BCEWithLogitsLoss applies sigmoid internally,
        # so there's no sigmoid here. Keep it that way; a double-sigmoid bug
        # (one here, one in the loss) is a classic silent-training-failure trap.
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)
        self._init_weights(init_scheme)

    def _init_weights(self, scheme: str):
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                if scheme == "zero":
                    nn.init.zeros_(m.weight)
                elif scheme == "xavier":
                    nn.init.xavier_normal_(m.weight)
                elif scheme == "he":
                    nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                else:
                    raise ValueError(f"Unknown init_scheme: {scheme}")
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def _to_tensor(X, y=None):
    X_t = torch.tensor(np.asarray(X), dtype=torch.float32)
    if y is None:
        return X_t
    y_t = torch.tensor(np.asarray(y), dtype=torch.float32)
    return X_t, y_t


def predict_proba(model, X):
    """Sigmoid is applied HERE, at inference — never inside the model (see 4.1 notes)."""
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(_to_tensor(X))).numpy()


def evaluate(model, X, y, threshold=0.5):
    """Metrics that matter for an imbalanced target — accuracy is included
    for reference only, never as the deciding number (EDA 2.1)."""
    probs = predict_proba(model, X)
    preds = (probs >= threshold).astype(int)
    y_arr = np.asarray(y)
    return {
        "accuracy": accuracy_score(y_arr, preds),
        "precision": precision_score(y_arr, preds, zero_division=0),
        "recall": recall_score(y_arr, preds, zero_division=0),
        "f1": f1_score(y_arr, preds, zero_division=0),
        "roc_auc": roc_auc_score(y_arr, probs),
        "pr_auc": average_precision_score(y_arr, probs),
    }


def tune_threshold(model, X, y, metric="f1"):
    """NEW. Sweep the decision threshold and return the one that maximises `metric`.

    ALWAYS pass the VALIDATION set here, never the test set. The first version of
    this notebook tuned the threshold directly on the test set and then reported
    test metrics at that threshold — that is a (subtle but real) form of test-set
    leakage: the threshold is a fitted parameter, so fitting it on test makes the
    reported test score optimistic. Fixed in 9.3.
    """
    probs = predict_proba(model, X)
    y_arr = np.asarray(y)
    scorer = {"f1": f1_score, "precision": precision_score, "recall": recall_score}[metric]
    grid = np.arange(0.05, 0.96, 0.01)
    scores = [scorer(y_arr, (probs >= t).astype(int), zero_division=0) for t in grid]
    best_i = int(np.argmax(scores))
    return float(grid[best_i]), float(scores[best_i])


_OPTIMIZERS = {
    "sgd": lambda params, lr, wd: torch.optim.SGD(params, lr=lr, weight_decay=wd),
    "momentum": lambda params, lr, wd: torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=wd),
    "rmsprop": lambda params, lr, wd: torch.optim.RMSprop(params, lr=lr, weight_decay=wd),
    "adam": lambda params, lr, wd: torch.optim.Adam(params, lr=lr, weight_decay=wd),
    "adamw": lambda params, lr, wd: torch.optim.AdamW(params, lr=lr, weight_decay=wd),
}


def train_model(model, X_train, y_train, X_val, y_val, epochs=25, lr=1e-3, batch_size=256,
                weight_decay=0.0, pos_weight=None, optimizer_name="adam",
                early_stopping=False, patience=5, monitor="val_loss",
                log_epoch_metrics=True, verbose=False):
    """Generic training loop, reused unchanged from here through Section 9.

    CHANGED vs. the first version of this notebook:
      * logs per-epoch metrics to MLflow (step=epoch) so the UI draws curves;
      * `history` now records val precision/recall/f1/pr_auc per epoch, not just losses;
      * `monitor` selects the early-stopping signal: "val_loss" (lower is better)
        or "val_pr_auc" / "val_f1" (higher is better).
    """
    X_t, y_t = _to_tensor(X_train, y_train)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)

    pw = torch.tensor(pos_weight, dtype=torch.float32) if pos_weight is not None else None
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    optimizer = _OPTIMIZERS[optimizer_name](model.parameters(), lr, weight_decay)

    X_val_t, y_val_t = _to_tensor(X_val, y_val)
    y_val_arr = np.asarray(y_val)
    history = {k: [] for k in
               ["train_loss", "val_loss", "val_precision", "val_recall", "val_f1", "val_pr_auc"]}

    higher_is_better = monitor != "val_loss"
    best_score = -np.inf if higher_is_better else np.inf
    best_state, best_epoch, epochs_no_improve = None, 0, 0
    in_mlflow_run = mlflow.active_run() is not None

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
        train_loss = running_loss / len(loader.dataset)

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_loss = criterion(val_logits, y_val_t).item()
            val_probs = torch.sigmoid(val_logits).numpy()
        val_preds = (val_probs >= 0.5).astype(int)

        epoch_metrics = {
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_precision": precision_score(y_val_arr, val_preds, zero_division=0),
            "val_recall": recall_score(y_val_arr, val_preds, zero_division=0),
            "val_f1": f1_score(y_val_arr, val_preds, zero_division=0),
            "val_pr_auc": average_precision_score(y_val_arr, val_probs),
        }
        for k, v in epoch_metrics.items():
            history[k].append(v)

        # This is what makes MLflow show a CURVE instead of a single point.
        if log_epoch_metrics and in_mlflow_run:
            mlflow.log_metrics(epoch_metrics, step=epoch)

        if verbose and (epoch % 5 == 0 or epoch == epochs - 1):
            print(f"  epoch {epoch:3d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} "
                  f"| val_P {epoch_metrics['val_precision']:.3f} | val_R {epoch_metrics['val_recall']:.3f} "
                  f"| val_F1 {epoch_metrics['val_f1']:.3f} | val_PR-AUC {epoch_metrics['val_pr_auc']:.3f}")

        if early_stopping:
            score = epoch_metrics[monitor]
            improved = (score > best_score + 1e-4) if higher_is_better else (score < best_score - 1e-4)
            if improved:
                best_score, best_epoch, epochs_no_improve = score, epoch, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    if verbose:
                        print(f"  early stopping at epoch {epoch} — no {monitor} improvement for "
                              f"{patience} epochs (best {monitor}={best_score:.4f} @ epoch {best_epoch})")
                    break

    if early_stopping and best_state is not None:
        model.load_state_dict(best_state)  # roll back to the best checkpoint, not the last one

    history["best_epoch"] = best_epoch
    history["best_score"] = best_score
    history["monitor"] = monitor
    return history


def summarise_history(name, history, metrics):
    """Compact one-line summary that shows the ACTUAL train and val loss values,
    not just the gap between them — so it's obvious which one is moving and in
    which direction."""
    tl, vl = history["train_loss"], history["val_loss"]
    min_val_epoch = int(np.argmin(vl))
    return (f"{name:20s} epochs={len(tl):3d} | train_loss {tl[0]:.4f}->{tl[-1]:.4f} "
            f"| val_loss {vl[0]:.4f}->{vl[-1]:.4f} | min val_loss {min(vl):.4f}@ep{min_val_epoch} "
            f"| gap {vl[-1]-tl[-1]:+.4f} | P {metrics['precision']:.3f} R {metrics['recall']:.3f} "
            f"F1 {metrics['f1']:.3f} PR-AUC {metrics['pr_auc']:.3f}")


print("Shared model class + train_model() + evaluate() + tune_threshold() ready.")

### 4.2 Four strategies, trained identically except for how they see the imbalance

Same architecture, same 25 epochs, same optimizer, every time — the *only* thing that changes between runs
is how class imbalance is handled. That's what makes the comparison fair; changing the model too would
confound the result.

**Critical rule: resampling only ever touches the training set.** `X_val`/`y_val` stay exactly as EDA found
them (~22% default) for every run below — validating against an artificially rebalanced set would tell you
how good the model is at the *rebalanced* problem, not the real one.

**Changed here:** each strategy is now also evaluated at its own *tuned* threshold (tuned on validation),
not only at the arbitrary 0.5 default. 4.3 explains why that changes the answer.

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

input_dim = X_train_scaled.shape[1]
n_pos = int(y_train.sum())
n_neg = int(len(y_train) - n_pos)
class_weight_pos_weight = n_neg / n_pos  # BCEWithLogitsLoss pos_weight: how many negatives per positive

X_train_smote, y_train_smote = SMOTE(random_state=RANDOM_STATE).fit_resample(X_train_scaled, y_train)
X_train_under, y_train_under = RandomUnderSampler(random_state=RANDOM_STATE).fit_resample(X_train_scaled, y_train)

print(f"Original train: {len(y_train)} rows, {y_train.mean():.1%} default")
print(f"SMOTE train:     {len(y_train_smote)} rows, {y_train_smote.mean():.1%} default")
print(f"Undersampled:    {len(y_train_under)} rows, {y_train_under.mean():.1%} default "
      f"(discards {len(y_train) - len(y_train_under)} rows = "
      f"{(len(y_train) - len(y_train_under)) / len(y_train):.1%} of the training data)")
print(f"class-weighted pos_weight = {class_weight_pos_weight:.2f} (no resampling, loss reweighted instead)")
print()

strategies = {
    "baseline":       dict(X=X_train_scaled, y=y_train,        pos_weight=None),
    "class_weighted": dict(X=X_train_scaled, y=y_train,        pos_weight=class_weight_pos_weight),
    "smote":          dict(X=X_train_smote,  y=y_train_smote,  pos_weight=None),
    "undersampled":   dict(X=X_train_under,  y=y_train_under,  pos_weight=None),
}

imbalance_results = {}
for name, cfg in strategies.items():
    torch.manual_seed(RANDOM_STATE)  # same starting weights across strategies, for a fair comparison
    model = CreditDefaultMLP(input_dim=input_dim)
    with mlflow.start_run(run_name=f"imbalance-{name}"):
        mlflow.log_params({"strategy": name, "epochs": 25, "lr": 1e-3, "optimizer": "adam",
                           "hidden_dims": "64,32", "train_rows": len(cfg["y"])})
        history = train_model(model, cfg["X"], cfg["y"], X_val_scaled, y_val,
                              epochs=25, pos_weight=cfg["pos_weight"])

        # Always scored on the untouched, still-imbalanced validation set.
        m_at_half = evaluate(model, X_val_scaled, y_val)                       # @ the arbitrary 0.5
        t_star, _ = tune_threshold(model, X_val_scaled, y_val, metric="f1")    # tuned on VALIDATION
        m_at_tuned = evaluate(model, X_val_scaled, y_val, threshold=t_star)

        mlflow.log_metrics(m_at_half)
        mlflow.log_metrics({f"tuned_{k}": v for k, v in m_at_tuned.items()})
        mlflow.log_metric("tuned_threshold", t_star)

    imbalance_results[name] = {
        **m_at_half,
        "threshold": t_star,
        "precision_tuned": m_at_tuned["precision"],
        "recall_tuned": m_at_tuned["recall"],
        "f1_tuned": m_at_tuned["f1"],
    }
    print(summarise_history(name, history, m_at_half))
    print(f"{'':20s} tuned threshold={t_star:.2f} -> P {m_at_tuned['precision']:.3f} "
          f"R {m_at_tuned['recall']:.3f} F1 {m_at_tuned['f1']:.3f}")

In [ ]:
results_df = pd.DataFrame(imbalance_results).T

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

results_df[["precision", "recall", "f1"]].plot(kind="bar", ax=axes[0],
                                               color=["#4C72B0", "#C44E52", "#55A868"])
axes[0].set_title("At the default threshold (0.5)")
axes[0].set_ylabel("Score (validation)")
axes[0].set_xticklabels(results_df.index, rotation=20, ha="right")

results_df[["precision_tuned", "recall_tuned", "f1_tuned"]].plot(kind="bar", ax=axes[1],
                                                                 color=["#4C72B0", "#C44E52", "#55A868"])
axes[1].set_title("At each strategy's own tuned threshold")
axes[1].set_xticklabels(results_df.index, rotation=20, ha="right")

# PR-AUC is threshold-INDEPENDENT: it measures how well the model ranks customers
# by risk, regardless of where you later draw the decision line. That makes it the
# fair way to ask "which strategy produced the better model?"
results_df["pr_auc"].plot(kind="bar", ax=axes[2], color="#8172B2")
axes[2].axhline(y_val.mean(), color="gray", linestyle="--", linewidth=1,
                label=f"random guess ({y_val.mean():.2f})")
axes[2].set_title("PR-AUC (threshold-independent)")
axes[2].set_xticklabels(results_df.index, rotation=20, ha="right")
axes[2].legend()

plt.tight_layout()
plt.show()

results_df[["precision", "recall", "f1", "threshold", "precision_tuned", "recall_tuned", "f1_tuned", "pr_auc"]].round(3)

**What the run actually showed** (first version of this notebook, F1 at threshold 0.5 only):

| Strategy | Precision | Recall | F1@0.5 | PR-AUC | Train rows |
|---|---|---|---|---|---|
| `baseline` | **0.634** | 0.396 | 0.488 | 0.544 | 21,000 |
| `class_weighted` | 0.446 | 0.610 | 0.515 | 0.536 | 21,000 |
| `smote` | 0.433 | **0.632** | 0.514 | **0.548** | 32,710 |
| `undersampled` | 0.459 | 0.603 | **0.521** | 0.529 | **9,290** |

Read the precision and recall columns as a pair and the pattern is unmistakable: **every strategy trades
precision for recall, and almost nothing else.** `baseline` has by far the best precision (0.634) and the
worst recall (0.396); `smote` is the mirror image (0.433 / 0.632). Nobody is "better" — they're sitting at
different points on the same curve.

And that's confirmed by the last column: **PR-AUC spans only 0.529–0.548 across all four strategies — a
spread of 0.019.** PR-AUC doesn't care where the decision threshold sits, so it measures the thing that
actually differs between models: how well they *rank* customers by risk. The answer is that all four rank
about equally well. Resampling didn't make the model smarter — it moved where the model draws the line.

**Which makes the original selection rule actively wrong.** The first version picked the winner with
`results_df["f1"].idxmax()`, which chose `undersampled` — the strategy that is **dead last on PR-AUC** and
that **discarded 55.8% of the training data** (21,000 → 9,290 rows), on the strength of an F1 lead of 0.006
over `class_weighted`. That margin is noise. Every experiment in Sections 6–9 then inherited a model
trained on less than half the available data.

That's the single most consequential bug this pass fixes, and 4.3 is the fix.

### 4.3 Choosing a strategy — and why F1 at threshold 0.5 is the wrong criterion

The trap in one sentence: **resampling shifts the model's predicted probabilities, so comparing strategies
at a fixed 0.5 threshold measures where each strategy happened to land relative to 0.5, not how good each
model is.**

`baseline` trains on data that is 78% non-default, so it learns to output low probabilities; at a 0.5 cut
it only flags the most obvious cases (high precision 0.634, low recall 0.396). `smote` trains on a 50/50
set, so its probabilities sit much higher and 0.5 flags far more people. Same underlying ranking ability
(PR-AUC 0.544 vs 0.548 — effectively tied), completely different F1@0.5.

**So: select on PR-AUC, then tune the threshold separately.** PR-AUC answers "which model ranks best",
threshold tuning answers "where do we draw the line" — two different questions that F1@0.5 jams together
and answers badly.

A tie-breaker also matters here, since the PR-AUC spread is small enough to be within noise: prefer the
strategy that keeps the most training data and adds the fewest assumptions. On that basis `baseline` and
`class_weighted` are both preferable to `undersampled` (throws away 56% of the data) and to `smote`
(fabricates 11,710 synthetic customers by interpolating between real ones — plausible for continuous
features, questionable for the one-hot `EDUCATION_*`/`MARRIAGE_*` columns, where an interpolated value like
0.4 doesn't correspond to any real person).

In [ ]:
# CHANGED: was `chosen_strategy = results_df["f1"].idxmax()`, which selected on
# F1 at the arbitrary 0.5 threshold and picked `undersampled` (last on PR-AUC,
# 56% of training data discarded). Now selects on PR-AUC — threshold-independent,
# so it measures ranking quality rather than threshold placement.
PR_AUC_NOISE_BAND = 0.01  # strategies within this of the best are treated as tied

best_pr_auc = results_df["pr_auc"].max()
contenders = results_df[results_df["pr_auc"] >= best_pr_auc - PR_AUC_NOISE_BAND]

print(f"Best PR-AUC: {best_pr_auc:.4f}")
print(f"Within the {PR_AUC_NOISE_BAND} noise band, i.e. statistically tied: {list(contenders.index)}")

# Tie-break among the contenders: prefer whichever keeps the most real training
# data and invents the least. Ranked by preference, first match wins.
PREFERENCE_ORDER = ["class_weighted", "baseline", "smote", "undersampled"]
chosen_strategy = next(s for s in PREFERENCE_ORDER if s in contenders.index)

chosen_cfg = strategies[chosen_strategy]
X_train_chosen, y_train_chosen = chosen_cfg["X"], chosen_cfg["y"]
pos_weight_chosen = chosen_cfg["pos_weight"]

print(f"\nChosen strategy: '{chosen_strategy}'")
print(f"  PR-AUC        : {results_df.loc[chosen_strategy, 'pr_auc']:.4f}")
print(f"  training rows : {len(y_train_chosen)}")
print(f"  pos_weight    : {pos_weight_chosen}")
print("\nUsed for every experiment from here on.")

## 5. MLP Model (PyTorch)

The architecture, formally. (The class itself had to be defined back in 4.1 to run the imbalance
comparison — this section is where its design choices get justified.)

In [ ]:
# CreditDefaultMLP was already defined in Section 4.1 (it had to be, to run the
# imbalance comparison) — this cell just instantiates the default config and
# inspects it, as the formal "here is the architecture" checkpoint for the rest
# of the notebook. Sections 6-9 reuse this same class with different config values.

default_model = CreditDefaultMLP(input_dim=input_dim)
print(default_model)

n_params = sum(p.numel() for p in default_model.parameters())
n_trainable = sum(p.numel() for p in default_model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {n_params:,} (trainable: {n_trainable:,})")
print(f"\nConfig (this is what becomes model_config.json in Section 10):\n{default_model.config}")

**Design decisions, and why:**

- **`hidden_dims=(64, 32)` — two hidden layers, narrowing.** Deep enough to combine features non-linearly
  (the reason to use an MLP over logistic regression at all), narrowing toward the output is the standard
  "funnel" shape; wide/deep options get their own fair test in Section 8, not guessed here.
- **`ReLU` activations.** Cheap, doesn't saturate for positive inputs (avoids the vanishing-gradient failure
  mode `sigmoid`/`tanh` are prone to in deeper nets) — which is also *why* `init_scheme="he"` is the default:
  He initialization is derived specifically for ReLU's variance behavior (Section 6 shows what happens with
  the wrong pairing).
- **`BatchNorm1d` after every `Linear`, before the activation.** Normalizes each layer's input distribution
  during training, which generally lets you train faster/more stably and acts as a mild regularizer —
  ablated explicitly in Section 7.
- **`Dropout` after every activation.** The other regularizer under test in Section 7 — randomly zeroing
  units during training so the network can't over-rely on any single one.
- **Output is a raw logit, no final sigmoid.** `nn.BCEWithLogitsLoss` (used in `train_model`) combines
  sigmoid + binary cross-entropy in one numerically-stable operation. Applying `torch.sigmoid()` is done
  separately at *inference* time (see `evaluate()` in 4.1) — never inside the model itself.

## 6. Weight Initialization Experiments

Zero vs. Xavier/Glorot vs. He, everything else held constant.

In [ ]:
init_schemes = ["zero", "xavier", "he"]
init_histories = {}
init_results = {}

for scheme in init_schemes:
    torch.manual_seed(RANDOM_STATE)
    model = CreditDefaultMLP(input_dim=input_dim, init_scheme=scheme)
    with mlflow.start_run(run_name=f"init-{scheme}"):
        mlflow.log_params({"init_scheme": scheme, "strategy": chosen_strategy, "epochs": 25})
        history = train_model(model, X_train_chosen, y_train_chosen, X_val_scaled, y_val,
                              epochs=25, pos_weight=pos_weight_chosen)
        metrics = evaluate(model, X_val_scaled, y_val)
        mlflow.log_metrics(metrics)
    init_histories[scheme] = history
    init_results[scheme] = metrics
    print(summarise_history(scheme, history, metrics))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# CHANGED: now plots train AND val loss for each scheme (the first version showed
# train loss only), so a diverging pair is visible rather than inferred.
for scheme, style in zip(init_schemes, ["-", "--", "-."]):
    axes[0].plot(init_histories[scheme]["train_loss"], style, label=f"{scheme} (train)")
axes[0].set_title("Training loss by weight init")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCE loss")
axes[0].legend()

for scheme, style in zip(init_schemes, ["-", "--", "-."]):
    axes[1].plot(init_histories[scheme]["val_pr_auc"], style, label=f"{scheme}")
axes[1].axhline(y_val.mean(), color="gray", linestyle=":", label="random guess")
axes[1].set_title("Validation PR-AUC by weight init")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("PR-AUC")
axes[1].legend()

init_results_df = pd.DataFrame(init_results).T
init_results_df[["precision", "recall", "f1"]].plot(kind="bar", ax=axes[2],
                                                    color=["#4C72B0", "#C44E52", "#55A868"])
axes[2].set_title("Final validation metrics by weight init")
axes[2].set_xticklabels(init_schemes, rotation=0)

plt.tight_layout()
plt.show()

init_results_df.round(3)

**What the run actually showed:**

| Init | final train_loss | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|
| `zero` | 0.6932 | 0.000 | 0.000 | 0.000 | **0.500** |
| `xavier` | 0.5584 | 0.466 | 0.598 | 0.524 | 0.775 |
| `he` | 0.5624 | 0.459 | 0.603 | 0.521 | 0.773 |

**`zero` is a textbook-perfect failure, and worth dwelling on.** Look at the exact numbers: train_loss
0.6932 is ln(2) = 0.6931 — the loss of a model that outputs probability 0.5 for every single input and has
learned nothing at all. ROC-AUC 0.500 is literally a coin flip. Precision, recall and F1 are all exactly
0.000 because the model never predicts the positive class for anyone.

The cause is structural, not bad luck. With every weight initialized to 0, every neuron in a layer computes
the identical output and therefore receives the identical gradient during backprop — so they update
identically, forever. They never differentiate from each other, and a 64-unit layer has the expressive
power of a single unit. This is the **symmetry-breaking problem**, and it's why *any* sensible
initialization is random.

**`xavier` (0.524) vs `he` (0.521) is a coin flip on this network** — a 0.003 F1 difference is well inside
run-to-run noise. That is the expected result at this depth: the two schemes differ in the variance they
scale the initial weights by, and that difference compounds *per layer*. With only 2 hidden layers there's
almost nothing to compound. He's advantage over Xavier is specifically derived for ReLU (which zeroes half
its inputs, halving the variance), and it shows up clearly in networks 10+ layers deep, not here.

**Don't over-read the fact that Xavier edged ahead.** `he` remains the default in `CreditDefaultMLP`
because the theory matches our activation function; the experiment shows the choice doesn't matter much at
this scale, which is a legitimate and useful finding in itself.

## 7. Regularization Experiments

Dropout, batch norm, L2 weight decay and early stopping — ablated individually, then all together.
40 epochs (up from 25) so there's room for the unregularized run to visibly overfit.

In [ ]:
reg_configs = {
    "none":                dict(dropout=0.0, use_batchnorm=False, weight_decay=0.0,  early_stopping=False),
    "dropout_only":        dict(dropout=0.3, use_batchnorm=False, weight_decay=0.0,  early_stopping=False),
    "batchnorm_only":      dict(dropout=0.0, use_batchnorm=True,  weight_decay=0.0,  early_stopping=False),
    "weight_decay_only":   dict(dropout=0.0, use_batchnorm=False, weight_decay=1e-2, early_stopping=False),
    "early_stopping_only": dict(dropout=0.0, use_batchnorm=False, weight_decay=0.0,  early_stopping=True),
    "all_combined":        dict(dropout=0.3, use_batchnorm=True,  weight_decay=1e-2, early_stopping=True),
}
# CHANGED: weight_decay bumped 1e-4 -> 1e-2. At 1e-4 the run was indistinguishable
# from no regularization at all (gap +0.0569 vs +0.0574 for `none`) — the penalty
# was too small to do anything, so the ablation taught nothing. 1e-2 is strong
# enough to actually show up.

reg_histories = {}
reg_results = {}

for name, cfg in reg_configs.items():
    torch.manual_seed(RANDOM_STATE)
    model = CreditDefaultMLP(input_dim=input_dim, dropout=cfg["dropout"], use_batchnorm=cfg["use_batchnorm"])
    with mlflow.start_run(run_name=f"reg-{name}"):
        mlflow.log_params({**cfg, "strategy": chosen_strategy, "epochs": 40})
        history = train_model(model, X_train_chosen, y_train_chosen, X_val_scaled, y_val,
                              epochs=40, weight_decay=cfg["weight_decay"], pos_weight=pos_weight_chosen,
                              early_stopping=cfg["early_stopping"], patience=5)
        metrics = evaluate(model, X_val_scaled, y_val)
        mlflow.log_metrics(metrics)
    reg_histories[name] = history
    reg_results[name] = metrics
    # CHANGED: prints the actual train and val loss values (start -> end) instead of
    # only their difference, so it's clear which curve moved and in which direction.
    print(summarise_history(name, history, metrics))

In [ ]:
# CHANGED: the first version plotted only 2 of the 6 configurations. All six now
# get their own panel with BOTH curves, so every ablation can be compared directly.
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True, sharey=True)

for ax, (name, history) in zip(axes.flat, reg_histories.items()):
    ax.plot(history["train_loss"], label="train", color="#4C72B0")
    ax.plot(history["val_loss"], label="val", color="#C44E52")
    final_gap = history["val_loss"][-1] - history["train_loss"][-1]
    ax.set_title(f"{name}\n{len(history['train_loss'])} epochs, final gap {final_gap:+.4f}", fontsize=10)
    ax.set_xlabel("Epoch"); ax.set_ylabel("BCE loss")
    ax.legend(fontsize=8)

plt.suptitle("Train vs. validation loss for every regularization configuration", y=1.00)
plt.tight_layout()
plt.show()

reg_results_df = pd.DataFrame(reg_results).T
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
gaps = {n: h["val_loss"][-1] - h["train_loss"][-1] for n, h in reg_histories.items()}
pd.Series(gaps).plot(kind="bar", ax=axes[0], color="#C44E52")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Final generalization gap (val_loss - train_loss)\nlower = less overfitting")
axes[0].set_xticklabels(list(gaps), rotation=45, ha="right")

reg_results_df[["precision", "recall", "f1"]].plot(kind="bar", ax=axes[1],
                                                   color=["#4C72B0", "#C44E52", "#55A868"])
axes[1].set_title("Final validation metrics")
axes[1].set_xticklabels(reg_results_df.index, rotation=45, ha="right")
plt.tight_layout()
plt.show()

reg_results_df.round(3)

**What the run actually showed** (with the original `weight_decay=1e-4`):

| Config | epochs | gap (val - train) | F1 |
|---|---|---|---|
| `none` | 40 | +0.0574 | 0.509 |
| `dropout_only` | 40 | **+0.0067** | 0.518 |
| `batchnorm_only` | 40 | **+0.0691** ⚠️ | 0.513 |
| `weight_decay_only` | 40 | +0.0569 | 0.511 |
| `early_stopping_only` | 22 | +0.0312 | 0.520 |
| `all_combined` | 13 | **-0.0049** | 0.519 |

**Dropout is the clear winner at closing the gap** — +0.0574 → +0.0067, a 9× reduction. It's doing exactly
what it's supposed to: forcing the network to spread its representation across units instead of letting a
few memorize training rows.

**Batch norm made overfitting *worse*, not better (+0.0691 vs `none`'s +0.0574).** This contradicts the
common "batch norm is also a regularizer" claim, and the first version of this notebook repeated that claim
uncritically — it's now corrected. What's going on: batch norm's regularizing effect comes from the noise
of per-batch statistics, but its *primary* effect is making optimization easier, which lets the network fit
the training set faster and harder. With batch=256 the batch statistics are stable enough that there's very
little noise to regularize with, so you get the faster-fitting without the noise benefit. Keep batch norm
for the training stability, don't count it as regularization.

**Weight decay at 1e-4 did nothing at all** (+0.0569 vs +0.0574 — a difference of 0.0005). The penalty was
simply too small relative to the loss to influence the weights. This is why the cell above now uses 1e-2:
an ablation that changes nothing teaches nothing. Expect it to actually bite at the new value.

**`all_combined`'s gap is *negative* (-0.0049), and that's not a bug.** When dropout is active, training
loss is measured *with* units randomly zeroed while validation loss is measured with the full network. The
model is being handicapped during the train measurement and not during the val measurement, so val can
legitimately come out lower. A small negative gap with dropout on is normal and healthy — don't
"fix" it.

**And the punchline: F1 barely moved (0.509 → 0.520 across all six).** Regularization tightened the loss
curves beautifully and bought roughly 0.01 F1. That's the honest lesson — on this dataset the model is not
capacity-limited, so regularization mostly buys stability and reproducibility, not accuracy. The real gains
came from feature engineering (Section 3) and will come from threshold tuning (9.3).

## 8. Hyperparameter Tuning

Grid search, random search and Optuna — same trial budget each, so the comparison is about search
*efficiency* rather than who got more attempts.

### 8.1 Shared trial runner

In [ ]:
# Shared trial-runner so grid/random/Optuna all score candidates identically —
# same epoch budget, same data, same MLflow logging shape. Only how each method
# *picks* the next candidate differs, which is the whole point of the comparison.
HPO_EPOCHS = 20


def run_trial(hidden_dims, dropout, lr, weight_decay, optimizer_name, batch_size, run_name):
    torch.manual_seed(RANDOM_STATE)
    model = CreditDefaultMLP(input_dim=input_dim, hidden_dims=hidden_dims, dropout=dropout)
    params = {
        "hidden_dims": str(hidden_dims), "dropout": dropout, "lr": lr,
        "weight_decay": weight_decay, "optimizer": optimizer_name, "batch_size": batch_size,
    }
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        train_model(model, X_train_chosen, y_train_chosen, X_val_scaled, y_val,
                    epochs=HPO_EPOCHS, lr=lr, batch_size=batch_size, weight_decay=weight_decay,
                    pos_weight=pos_weight_chosen, optimizer_name=optimizer_name,
                    early_stopping=True, patience=4, monitor="val_pr_auc")
        metrics = evaluate(model, X_val_scaled, y_val)
        # CHANGED: trials are now ranked by PR-AUC rather than F1@0.5, for the same
        # reason as 4.3 — F1@0.5 rewards a lucky threshold position, PR-AUC measures
        # how well the model actually ranks. The threshold gets tuned once, in 9.3.
        t_star, _ = tune_threshold(model, X_val_scaled, y_val, metric="f1")
        m_tuned = evaluate(model, X_val_scaled, y_val, threshold=t_star)
        mlflow.log_metrics(metrics)
        mlflow.log_metrics({f"tuned_{k}": v for k, v in m_tuned.items()})
        mlflow.log_metric("tuned_threshold", t_star)
    return {**params, **metrics, "f1_tuned": m_tuned["f1"], "threshold": t_star}


print(f"run_trial() ready — every HPO method below scores candidates over {HPO_EPOCHS} epochs, identically.")
print("Selection metric: PR-AUC (threshold-independent).")

### 8.2 Grid search — exhaustive over a small, deliberately coarse grid

Every combination of a handful of values for a few knobs. Guaranteed to check everywhere in the grid, but
the grid has to stay small — it grows multiplicatively with every dimension you add (3 learning rates × 2
architectures × 2 dropout rates = 12 trials; add one more 3-value dimension and it's 36). That combinatorial
cost is the whole reason random search and Optuna exist — see 8.3/8.4.

In [ ]:
import itertools

grid = {
    "lr": [1e-2, 1e-3, 1e-4],
    "hidden_dims": [(32,), (64, 32)],
    "dropout": [0.2, 0.4],
}
grid_combos = list(itertools.product(grid["lr"], grid["hidden_dims"], grid["dropout"]))
print(f"Grid search: {len(grid_combos)} trials ({' x '.join(str(len(v)) for v in grid.values())})")

grid_trials = []
for i, (lr, hidden_dims, dropout) in enumerate(grid_combos):
    result = run_trial(hidden_dims, dropout, lr, weight_decay=1e-4, optimizer_name="adam",
                       batch_size=256, run_name=f"grid-{i:02d}")
    grid_trials.append(result)
    print(f"  [{i:2d}/{len(grid_combos)}] lr={lr:<7} hidden={str(hidden_dims):<10} dropout={dropout} "
          f"-> PR-AUC={result['pr_auc']:.3f} F1@tuned={result['f1_tuned']:.3f}")

grid_trials_df = pd.DataFrame(grid_trials)
print("\nBest grid trial (by PR-AUC):")
print(grid_trials_df.loc[grid_trials_df["pr_auc"].idxmax()])

### 8.3 Random search — same trial budget, wider and continuous search space

Same number of trials as the grid (12), but instead of fixed grid points, each trial samples freely from a
*wider* space — including `optimizer` and `batch_size`, which the grid never touched at all. Classic
random-search result (Bergstra & Bengio, 2012): for a fixed budget, sampling randomly across more dimensions
usually beats a grid restricted to few dimensions, because most hyperparameters don't matter equally — grid
search wastes trials being exhaustive along low-impact dimensions.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
N_RANDOM_TRIALS = 12

hidden_dims_choices = [(32,), (64,), (64, 32), (128, 64), (64, 32, 16)]
optimizer_choices = ["sgd", "momentum", "rmsprop", "adam", "adamw"]
batch_size_choices = [64, 128, 256]

random_trials = []
for i in range(N_RANDOM_TRIALS):
    lr = 10 ** rng.uniform(-4, -1.5)  # log-uniform: HP search spaces for lr span orders of magnitude
    hidden_dims = hidden_dims_choices[rng.integers(len(hidden_dims_choices))]
    dropout = rng.uniform(0.1, 0.5)
    weight_decay = 10 ** rng.uniform(-6, -2)
    optimizer_name = optimizer_choices[rng.integers(len(optimizer_choices))]
    batch_size = int(batch_size_choices[rng.integers(len(batch_size_choices))])

    result = run_trial(hidden_dims, dropout, lr, weight_decay, optimizer_name, batch_size,
                       run_name=f"random-{i:02d}")
    random_trials.append(result)
    print(f"  [{i:2d}/{N_RANDOM_TRIALS}] lr={lr:.5f} hidden={str(hidden_dims):<14} dropout={dropout:.2f} "
          f"opt={optimizer_name:<9} batch={batch_size} -> PR-AUC={result['pr_auc']:.3f} "
          f"F1@tuned={result['f1_tuned']:.3f}")

random_trials_df = pd.DataFrame(random_trials)
print("\nBest random trial (by PR-AUC):")
print(random_trials_df.loc[random_trials_df["pr_auc"].idxmax()])

### 8.4 Optuna — same budget again, but each trial learns from the ones before it

Grid and random search both pick every trial's hyperparameters independently — trial #10 knows nothing
about how trials #1-9 scored. Optuna's default sampler (TPE — Tree-structured Parzen Estimator) builds a
probabilistic model of "which regions of the search space score well" as trials complete, and biases new
suggestions toward those regions (while still exploring). Same 12-trial budget as the other two, so the
comparison in 8.5 is about search *efficiency*, not who got more attempts.

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)  # the per-trial print below is enough noise already
optuna_trials = []


def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 3e-2, log=True)
    hidden_dims_str = trial.suggest_categorical("hidden_dims", [str(h) for h in hidden_dims_choices])
    hidden_dims = eval(hidden_dims_str)  # safe: only ever comes from our own fixed list above
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", optimizer_choices)
    batch_size = trial.suggest_categorical("batch_size", batch_size_choices)

    result = run_trial(hidden_dims, dropout, lr, weight_decay, optimizer_name, batch_size,
                       run_name=f"optuna-{trial.number:02d}")
    optuna_trials.append(result)
    print(f"  [{trial.number:2d}] PR-AUC={result['pr_auc']:.3f} F1@tuned={result['f1_tuned']:.3f}")
    return result["pr_auc"]  # CHANGED: optimise PR-AUC, matching 8.1's selection metric


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=12, show_progress_bar=False)

optuna_trials_df = pd.DataFrame(optuna_trials)
print(f"\nBest Optuna trial: PR-AUC={study.best_value:.3f}")
print(study.best_params)

### 8.5 Which search strategy used its budget best?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for label, trials_df in [("grid", grid_trials_df), ("random", random_trials_df), ("optuna", optuna_trials_df)]:
    best_so_far = trials_df["pr_auc"].cummax()
    ax.plot(range(1, len(best_so_far) + 1), best_so_far, marker="o", label=label)
ax.set_xlabel("Trial number")
ax.set_ylabel("Best validation PR-AUC so far")
ax.set_title("Search efficiency: best-so-far PR-AUC per trial, same 12-trial budget each")
ax.legend()
plt.tight_layout()
plt.show()

all_trials_df = pd.concat([
    grid_trials_df.assign(method="grid"),
    random_trials_df.assign(method="random"),
    optuna_trials_df.assign(method="optuna"),
], ignore_index=True)

best_trial = all_trials_df.loc[all_trials_df["pr_auc"].idxmax()]
print(f"Overall best trial (method={best_trial['method']}, PR-AUC={best_trial['pr_auc']:.3f}):")
print(best_trial)

**What the run actually showed** (when trials were still ranked by F1@0.5):

- **Random search won**, F1 0.532, with `(64, 32, 16)` + `adamw` + lr 0.00128 + batch 64.
- **Grid search**: best 0.523 — and note *seven of its twelve trials landed between 0.514 and 0.523*. The
  grid spent most of its budget re-confirming that lr 1e-2 and 1e-3 both work fine.
- **Optuna: 0.522 — it lost.**

**The grid's real failure is visible in its worst trials:** all four `lr=1e-4` runs scored 0.446–0.478,
far below everything else. That's a third of the entire grid budget spent confirming a learning rate that
was obviously too low after the first trial. Grid search cannot learn that and skip ahead — it is
committed to the full product of its axes. Random search, sampling `lr` log-uniformly, only wasted a
couple of trials down there.

**Why Optuna lost, and why that's not an indictment of Optuna:** TPE needs a warm-up period (roughly its
first 10 trials, by default) of essentially random sampling before its probabilistic model has enough
observations to guide anything. With a 12-trial budget it barely finishes warming up before the run ends —
so it was effectively doing random search, with slightly worse luck. Optuna starts paying off in the
50–100+ trial range. Increase `n_trials` if you ever want to see the real difference; at n=12 the honest
conclusion is **"all three methods are within noise of each other, and random search got the best draw."**

**Also worth noting: the entire search moved F1 from 0.521 to 0.532 — about 0.01.** Same story as
Section 7. Architecture and optimizer choices are not the bottleneck on this dataset.

## 9. Final Model Evaluation & Error Analysis

Retrain the winner properly, tune the decision threshold **on validation**, then evaluate once on the
held-out test set.

### 9.1 Retrain the winning configuration

Two changes here, both mattering more than they look:

**1. Early stopping now monitors validation PR-AUC, not validation loss.** In the original run the final
model stopped at **epoch 13 of a 100-epoch budget** — badly undertrained. The reason is that `val_loss` is
computed on the *reweighted* objective (with `pos_weight`, or on resampled data), so it stops improving
well before the model's actual *ranking* ability stops improving. We don't deploy the loss; we deploy the
ranking. So we stop on the thing we care about.

**2. Patience raised 10 → 20, epochs 100 → 200.** PR-AUC is noisier epoch-to-epoch than loss, so it needs
more patience to avoid stopping on a random dip. Since the best checkpoint is restored at the end, a
too-generous patience costs only time, never quality.

In [ ]:
final_hidden_dims = eval(best_trial["hidden_dims"]) if isinstance(best_trial["hidden_dims"], str) else best_trial["hidden_dims"]

torch.manual_seed(RANDOM_STATE)
final_model = CreditDefaultMLP(input_dim=input_dim, hidden_dims=final_hidden_dims,
                               dropout=float(best_trial["dropout"]))

with mlflow.start_run(run_name="final-model"):
    final_params = {
        "hidden_dims": str(final_hidden_dims), "dropout": float(best_trial["dropout"]),
        "lr": float(best_trial["lr"]), "weight_decay": float(best_trial["weight_decay"]),
        "optimizer": best_trial["optimizer"], "batch_size": int(best_trial["batch_size"]),
        "strategy": chosen_strategy, "source_search_method": best_trial["method"],
        "early_stopping_monitor": "val_pr_auc",
    }
    mlflow.log_params(final_params)
    final_history = train_model(
        final_model, X_train_chosen, y_train_chosen, X_val_scaled, y_val,
        epochs=200, lr=final_params["lr"], batch_size=final_params["batch_size"],
        weight_decay=final_params["weight_decay"], pos_weight=pos_weight_chosen,
        optimizer_name=final_params["optimizer"],
        early_stopping=True, patience=20, monitor="val_pr_auc", verbose=True,
    )

print(f"\nTrained {len(final_history['train_loss'])} epochs; "
      f"best val_pr_auc={final_history['best_score']:.4f} at epoch {final_history['best_epoch']} "
      f"(weights restored to that checkpoint).")

# The training curves for the final model — the plot the first version never drew.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(final_history["train_loss"], label="train", color="#4C72B0")
axes[0].plot(final_history["val_loss"], label="val", color="#C44E52")
axes[0].axvline(final_history["best_epoch"], color="black", linestyle="--", label="restored checkpoint")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCE loss"); axes[0].legend()

axes[1].plot(final_history["val_precision"], label="precision", color="#4C72B0")
axes[1].plot(final_history["val_recall"], label="recall", color="#C44E52")
axes[1].plot(final_history["val_f1"], label="F1", color="#55A868")
axes[1].set_title("Validation precision / recall / F1 @ 0.5"); axes[1].set_xlabel("Epoch"); axes[1].legend()

axes[2].plot(final_history["val_pr_auc"], color="#8172B2")
axes[2].axvline(final_history["best_epoch"], color="black", linestyle="--")
axes[2].set_title("Validation PR-AUC (the early-stopping signal)"); axes[2].set_xlabel("Epoch")

plt.tight_layout()
plt.show()

### 9.2 Selecting the best run programmatically from MLflow

Everything above tracked its own `best_trial` in Python, but the deliverable requirement is selecting the
winner *from MLflow* — the whole reason every run was logged. `mlflow.search_runs` queries the tracking
store directly, which is also how you'd revisit this after restarting the Colab runtime and losing the
Python variables (the logged runs persist in `mlruns/` regardless).

In [ ]:
all_runs = mlflow.search_runs(experiment_names=["credit-card-default"])

ranked = all_runs.sort_values("metrics.pr_auc", ascending=False)[
    ["tags.mlflow.runName", "metrics.pr_auc", "metrics.f1", "metrics.precision", "metrics.recall"]
]
print(f"Total runs logged this session: {len(all_runs)}")
print("\nTop 10 runs across the entire experiment, by validation PR-AUC:")
ranked.head(10)

### 9.3 Tune the decision threshold — on VALIDATION, not test

**This is the bug fix that matters most for the headline numbers.** The first version of this notebook
swept the threshold against `y_test`, picked the value that maximised test F1, and then reported test
metrics at that threshold. The threshold is a fitted parameter like any other, so fitting it on the test
set and scoring on the same set makes the reported score optimistic — the test set stops being held-out
the moment you fit anything to it.

The fix: tune on validation (which we've already used for model selection throughout, so nothing is lost),
freeze the result, and let the test set stay genuinely untouched until 9.4.

In [ ]:
val_probs = predict_proba(final_model, X_val_scaled)

thresholds = np.arange(0.05, 0.96, 0.01)
sweep = []
for t in thresholds:
    preds_t = (val_probs >= t).astype(int)
    sweep.append({
        "threshold": t,
        "precision": precision_score(y_val, preds_t, zero_division=0),
        "recall": recall_score(y_val, preds_t, zero_division=0),
        "f1": f1_score(y_val, preds_t, zero_division=0),
    })
sweep_df = pd.DataFrame(sweep)
best_threshold = float(sweep_df.loc[sweep_df["f1"].idxmax(), "threshold"])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sweep_df["threshold"], sweep_df["precision"], label="precision", color="#4C72B0")
ax.plot(sweep_df["threshold"], sweep_df["recall"], label="recall", color="#C44E52")
ax.plot(sweep_df["threshold"], sweep_df["f1"], label="f1", color="#55A868")
ax.axvline(0.5, color="gray", linestyle=":", label="default (0.5)")
ax.axvline(best_threshold, color="black", linestyle="--", label=f"chosen ({best_threshold:.2f})")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score")
ax.set_title("Precision / recall / F1 vs. decision threshold — VALIDATION set")
ax.legend()
plt.tight_layout()
plt.show()

val_at_half = evaluate(final_model, X_val_scaled, y_val, threshold=0.5)
val_at_tuned = evaluate(final_model, X_val_scaled, y_val, threshold=best_threshold)
print(f"Chosen threshold: {best_threshold:.2f} (tuned on VALIDATION, frozen from here on)")
print(f"  validation @0.50 -> P {val_at_half['precision']:.4f} R {val_at_half['recall']:.4f} F1 {val_at_half['f1']:.4f}")
print(f"  validation @{best_threshold:.2f} -> P {val_at_tuned['precision']:.4f} R {val_at_tuned['recall']:.4f} F1 {val_at_tuned['f1']:.4f}")

# --- Business-cost alternative (not used, but this is where you'd change it) ---
# Maximising F1 treats a false positive and a false negative as equally bad. In
# credit risk they usually aren't: a missed defaulter costs the outstanding
# balance, a wrongly-declined good customer costs the margin on one account.
# If a false negative is ~5x costlier than a false positive, you'd pick the
# threshold minimising (5 * FN + 1 * FP) instead of maximising F1 — which lands
# on a LOWER threshold, catching more defaulters at the price of more false alarms.
FN_COST, FP_COST = 5, 1
costs = [(t, FN_COST * ((val_probs < t) & (np.asarray(y_val) == 1)).sum()
             + FP_COST * ((val_probs >= t) & (np.asarray(y_val) == 0)).sum())
         for t in thresholds]
cost_optimal_threshold = min(costs, key=lambda kv: kv[1])[0]
print(f"\nFor reference — if false negatives were {FN_COST}x costlier than false positives,")
print(f"the cost-optimal threshold would be {cost_optimal_threshold:.2f} instead of {best_threshold:.2f}.")

**Reading this plot:** precision (blue) rises with the threshold, recall (red) falls — mechanically, a
higher bar means you flag fewer people, and the ones you do flag are more likely to be genuine defaulters.
F1 (green) peaks where the two are balanced.

**Set expectations honestly here.** In the original run the tuned threshold came out at 0.49 versus the
0.5 default, and F1 moved from 0.5340 to 0.5346 — a gain of **0.0006**, i.e. nothing. That is not a failure
of threshold tuning; it's a consequence of the *imbalance strategy* already having done the job. Training
on a rebalanced (50/50) set shifts the predicted probabilities so that 0.5 already sits near the optimum.
If you had trained on `baseline` (no imbalance handling), the probabilities would sit much lower, the
optimal threshold would land nearer 0.25–0.30, and tuning would be worth several points of F1.

So the two levers are substitutes, not additions: **handle the imbalance in the loss, or handle it in the
threshold — doing both well doesn't stack.** Which is worth knowing before you spend an afternoon expecting
them to.

### 9.4 Final test-set evaluation — the one and only look at the test set

In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, precision_recall_curve,
)

test_probs = predict_proba(final_model, X_test_scaled)
test_preds_default = (test_probs >= 0.5).astype(int)
test_preds_tuned = (test_probs >= best_threshold).astype(int)

test_metrics_default_threshold = evaluate(final_model, X_test_scaled, y_test, threshold=0.5)
final_test_metrics = evaluate(final_model, X_test_scaled, y_test, threshold=best_threshold)

with mlflow.start_run(run_name="final-model-test-eval"):
    mlflow.log_params({**final_params, "decision_threshold": best_threshold})
    mlflow.log_metrics({f"test_{k}": v for k, v in final_test_metrics.items()})
    mlflow.log_metrics({f"test_at_half_{k}": v for k, v in test_metrics_default_threshold.items()})
    mlflow.pytorch.log_model(final_model, "model")

print("Classification report at the tuned threshold:")
print(classification_report(y_test, test_preds_tuned, target_names=["No Default", "Default"]))

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

for ax, preds, title in [
    (axes[0], test_preds_default, "Confusion matrix @ 0.50 (default)"),
    (axes[1], test_preds_tuned, f"Confusion matrix @ {best_threshold:.2f} (tuned)"),
]:
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=["Pred: No Default", "Pred: Default"],
                yticklabels=["True: No Default", "True: Default"])
    ax.set_title(title)

fpr, tpr, _ = roc_curve(y_test, test_probs)
axes[2].plot(fpr, tpr, color="#4C72B0", label=f"AUC={final_test_metrics['roc_auc']:.3f}")
axes[2].plot([0, 1], [0, 1], "--", color="gray", label="random guess")
axes[2].set_xlabel("False Positive Rate"); axes[2].set_ylabel("True Positive Rate")
axes[2].set_title("ROC curve"); axes[2].legend()

prec, rec, _ = precision_recall_curve(y_test, test_probs)
axes[3].plot(rec, prec, color="#C44E52", label=f"AP={final_test_metrics['pr_auc']:.3f}")
axes[3].axhline(y_test.mean(), linestyle="--", color="gray", label=f"random guess ({y_test.mean():.2f})")
axes[3].set_xlabel("Recall"); axes[3].set_ylabel("Precision")
axes[3].set_title("Precision-Recall curve"); axes[3].legend()

plt.tight_layout()
plt.show()

print(f"{'metric':12s} {'@0.50':>10s} {'@tuned':>10s}")
for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]:
    print(f"{k:12s} {test_metrics_default_threshold[k]:10.4f} {final_test_metrics[k]:10.4f}")

**What the run actually showed** (test set, tuned threshold 0.49):

```
              precision    recall  f1-score   support
  No Default       0.87      0.84      0.85      3504
     Default       0.50      0.58      0.53       996
    accuracy                           0.78      4500
```

**Put the accuracy number in context immediately.** 0.78 accuracy looks respectable until you remember EDA
2.1: predicting "no default" for all 4,500 customers scores **0.7788**. The model's accuracy is
statistically indistinguishable from the do-nothing baseline. That is not a broken model — it's proof that
accuracy is the wrong metric here, exactly as 2.1 predicted. The model's value is entirely in the Default
row: it identifies 58% of actual defaulters, which the do-nothing baseline identifies 0% of.

**Reading the two confusion matrices side by side** shows the threshold trade in raw counts — moving the
threshold down converts some bottom-left cells (false negatives, missed defaulters) into top-right cells
(false positives, wrongly flagged good customers). Neither matrix is "correct"; which one you want depends
on the relative cost of those two mistakes, which is the calculation sketched at the end of 9.3.

**ROC-AUC 0.763 vs PR-AUC 0.515 — and PR-AUC is the honest one.** ROC's false-positive rate has the large
majority class (3,504 rows) in its denominator, so a fixed number of false positives barely moves it. PR
uses precision, whose denominator is only the flagged set, so it stays sensitive to the minority class.
The ~0.25 gap between the two numbers is the imbalance distorting ROC, not two different findings. Quote
PR-AUC in the writeup.

**Is F1 ≈ 0.53 for the default class good?** For this dataset, yes — it's in line with published results,
which cluster around 0.52–0.55 for the minority class regardless of model family (logistic regression,
gradient boosting and neural nets all land in the same band). The dataset has 23 features describing six
months of behavior, and that simply doesn't determine next-month default with high confidence. **The
ceiling here is a property of the data, not of the model.** Which is worth stating plainly in the writeup —
it's a stronger result than an unexplained number, and it's why Sections 7 and 8 moved F1 by ~0.01 each
while Section 3's feature engineering moved the strongest single predictor by 22%.

### 9.5 Error analysis — do the mistakes have a pattern?

In [ ]:
error_df = X_test.copy()  # unscaled, human-readable values — easier to eyeball than the scaled array
error_df["y_true"] = y_test.values
error_df["y_pred"] = test_preds_tuned
error_df["p_default"] = test_probs

false_negatives = error_df[(error_df.y_true == 1) & (error_df.y_pred == 0)]
false_positives = error_df[(error_df.y_true == 0) & (error_df.y_pred == 1)]
true_positives = error_df[(error_df.y_true == 1) & (error_df.y_pred == 1)]

print(f"False negatives: {len(false_negatives)}  |  False positives: {len(false_positives)}  "
      f"|  True positives: {len(true_positives)}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.kdeplot(true_positives["PAY_0"], label="True Positive (caught)", ax=axes[0],
            color="#55A868", fill=True, alpha=0.3)
sns.kdeplot(false_negatives["PAY_0"], label="False Negative (missed)", ax=axes[0],
            color="#C44E52", fill=True, alpha=0.3)
axes[0].set_title("PAY_0: caught vs. missed defaulters")
axes[0].legend()

# CHANGED: added DELINQUENCY_STREAK, since 3.3 showed it is the strongest single
# predictor — if the misses differ on anything, they should differ on this.
sns.kdeplot(true_positives["DELINQUENCY_STREAK"], label="True Positive (caught)", ax=axes[1],
            color="#55A868", fill=True, alpha=0.3)
sns.kdeplot(false_negatives["DELINQUENCY_STREAK"], label="False Negative (missed)", ax=axes[1],
            color="#C44E52", fill=True, alpha=0.3)
axes[1].set_title("DELINQUENCY_STREAK: caught vs. missed")
axes[1].legend()

sns.histplot(false_negatives["p_default"], bins=20, ax=axes[2], color="#C44E52")
axes[2].axvline(best_threshold, color="black", linestyle="--", label=f"threshold={best_threshold:.2f}")
axes[2].set_title("Predicted probability of the misses\n(how close were they?)")
axes[2].legend()

plt.tight_layout()
plt.show()

near_misses = (false_negatives["p_default"] >= best_threshold - 0.10).sum()
print(f"\nOf {len(false_negatives)} false negatives, {near_misses} "
      f"({near_misses / max(len(false_negatives), 1):.1%}) were within 0.10 of the threshold — "
      f"i.e. near-misses rather than confident errors.")
print("\nProfile comparison (mean values):")
print(pd.DataFrame({
    "caught (TP)": true_positives[["PAY_0", "DELINQUENCY_STREAK", "UTILIZATION", "LIMIT_BAL"]].mean(),
    "missed (FN)": false_negatives[["PAY_0", "DELINQUENCY_STREAK", "UTILIZATION", "LIMIT_BAL"]].mean(),
}).round(3))

**What the run actually showed:** 405 false negatives, 624 false positives, 591 true positives — so the
model catches roughly 59% of defaulters and, for every 100 people it flags, about 49 genuinely default.

**Read the two KDE plots as one question: are the misses a different kind of customer, or just borderline
cases of the same kind?**

If the red (missed) curves sit clearly to the left of the green (caught) curves on both `PAY_0` and
`DELINQUENCY_STREAK`, the answer is that **the model misses defaulters who had clean recent repayment
records** — people who paid on time for six months and then defaulted anyway. This is the important case,
because no amount of model tuning fixes it: the features genuinely do not contain the signal. Those
defaults are driven by things this dataset never recorded — job loss, medical bills, a new loan elsewhere.
The fix is more data (income changes, other credit lines, macro indicators), not a better MLP.

**The third plot separates "uncertain" from "wrong".** False negatives clustered just below the threshold
are near-misses — the model ranked them correctly as risky, the cut just fell on the wrong side. Those are
recoverable by lowering the threshold (at the cost of more false positives; see the cost calculation in
9.3). False negatives sitting down near probability 0.0 are the genuinely concerning ones: the model was
*confident* they were safe and it was wrong. The printed near-miss percentage tells you which population
dominates, and therefore whether threshold tuning or feature work is the right next investment.

**Note also that false positives (624) outnumber true positives (591).** At this threshold the model raises
slightly more false alarms than correct catches — which is the direct, unavoidable consequence of a
precision of ~0.49 on a 22%-prevalence problem. Worth being upfront about in the writeup rather than
letting a reader discover it in the confusion matrix.

## 10. Export Artifacts

Everything the FastAPI backend needs, and nothing it doesn't.

In [ ]:
import json
import pickle
import os

os.makedirs("export/model", exist_ok=True)
os.makedirs("export/preprocessing", exist_ok=True)
os.makedirs("export/metrics", exist_ok=True)

# --- model/model.pt: trained weights only (not the whole object — state_dict
#     is the portable, version-independent way to persist a PyTorch model) ---
torch.save(final_model.state_dict(), "export/model/model.pt")

# --- model/model_config.json: everything needed to reconstruct CreditDefaultMLP
#     and apply its predictions correctly, without needing this notebook again ---
model_config = {
    **final_model.config,
    "decision_threshold": best_threshold,
    "feature_columns": feature_columns,  # exact order the model expects at inference
    "imbalance_strategy": chosen_strategy,
    "training_hyperparams": {
        "lr": final_params["lr"], "weight_decay": final_params["weight_decay"],
        "optimizer": final_params["optimizer"], "batch_size": final_params["batch_size"],
    },
}
with open("export/model/model_config.json", "w") as f:
    json.dump(model_config, f, indent=2)

# --- preprocessing/scaler.pkl + feature_columns.json ---
with open("export/preprocessing/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("export/preprocessing/feature_columns.json", "w") as f:
    json.dump(feature_columns, f, indent=2)

# --- metrics/evaluation_report.json — for the README/docs, not needed at inference ---
evaluation_report = {
    "test_metrics_at_threshold_0.5": test_metrics_default_threshold,
    "test_metrics_at_chosen_threshold": final_test_metrics,
    "chosen_threshold": best_threshold,
    "naive_baseline_accuracy": float(naive_accuracy),  # from EDA 2.1 — the floor this model clears
    "imbalance_strategy": chosen_strategy,
    "hyperparameter_search_winner_method": best_trial["method"],
}
with open("export/metrics/evaluation_report.json", "w") as f:
    json.dump(evaluation_report, f, indent=2)

print("Exported to ./export/ :")
for root, _, filenames in os.walk("export"):
    for fn in filenames:
        print(" ", os.path.join(root, fn))

In [ ]:
# Zip both the export/ folder and mlruns/ for download in one go.
!zip -r export.zip export/ -x "*.DS_Store"
!zip -r mlruns_export.zip mlruns/ -x "*.DS_Store"

from google.colab import files

files.download("export.zip")
files.download("mlruns_export.zip")

### 10.1 After downloading — where things go locally

1. Unzip `export.zip` and copy its contents into `ml/artifacts/` in the repo, matching the same
   `model/`, `preprocessing/`, `metrics/` layout — these become the FastAPI backend's inputs in Stage 2.
2. Follow [mlflow-workflow.md](../../docs/mlflow-workflow.md) to merge `mlruns_export.zip` into
   `infra/mlflow/data/mlruns/`, then `make mlflow-up` + `make mlflow-ui` to browse the full experiment
   history — every run from Sections 4, 6, 7, 8, and this final one — locally.
3. Stage 1.5 is done once both of those are in place. Update `docs/progress.md`'s checklist, or just say so
   in chat — Stage 2 (FastAPI) picks up from here.

In [ ]:
import mlflow

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("credit-card-default")

with mlflow.start_run(run_name="baseline-mlp"):
    mlflow.log_params({"lr": 1e-3, "batch_size": 64, "optimizer": "adam"})
    mlflow.log_metrics({"val_precision": 0.71, "val_recall": 0.63, "val_f1": 0.67})
    mlflow.pytorch.log_model(model, "model")
    # ... or mlflow.log_artifact("scaler.pkl") for non-PyTorch artifacts

In [ ]:
mlflow.search_runs(experiment_names=["credit-card-default"]).sort_values("metrics.val_f1", ascending=False)